In [1]:
import pandas as pd


df = pd.read_json("output/results_raw_vllm/test_main_eval_stream_batch_Qwen__Qwen3-30B-A3B-Instruct-2507.jsonl", lines=True)

In [ ]:
import json
import textwrap
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, display


def to_int_or_none(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def parse_raw_output(raw_output):
    if isinstance(raw_output, dict):
        return raw_output
    if isinstance(raw_output, str) and raw_output.strip():
        try:
            parsed = json.loads(raw_output)
            if isinstance(parsed, dict):
                return parsed
            return {"parsed_output": parsed}
        except json.JSONDecodeError:
            return {"raw_output": raw_output}
    return {}


def normalize_text(value, default="<missing>"):
    if value is None:
        return default
    if isinstance(value, float) and pd.isna(value):
        return default
    text = str(value).strip()
    return text if text else default


def wrap_long_text(value, width=100):
    text = normalize_text(value)
    if text == "<missing>":
        return text

    wrapped_lines = []
    for line in str(text).splitlines() or [str(text)]:
        if line.strip():
            wrapped_lines.append(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
        else:
            wrapped_lines.append("")
    return "\n".join(wrapped_lines)


def load_mapping_records(mapping_path):
    if not mapping_path.exists():
        raise FileNotFoundError(f"Mapping file not found: {mapping_path}")

    raw_text = mapping_path.read_text(encoding="utf-8")

    try:
        parsed = json.loads(raw_text)
        if isinstance(parsed, list):
            return [row for row in parsed if isinstance(row, dict)]
        if isinstance(parsed, dict):
            if isinstance(parsed.get("data"), list):
                return [row for row in parsed["data"] if isinstance(row, dict)]
            if isinstance(parsed.get("records"), list):
                return [row for row in parsed["records"] if isinstance(row, dict)]
            return [parsed]
    except json.JSONDecodeError:
        pass

    records = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(row, dict):
            records.append(row)
    return records


def build_mapping_indices(records):
    map_by_row_idx = {}
    map_by_question_id = {}
    row_idx_conflicts = {}
    question_id_conflicts = {}

    for row in records:
        row_idx = to_int_or_none(row.get("row_idx"))
        question_id = to_int_or_none(row.get("question_id"))
        dataset_name = row.get("dataset")
        dataset_name = str(dataset_name) if dataset_name is not None else pd.NA

        meta = {
            "row_idx": row_idx,
            "source_question_id": question_id,
            "dataset": dataset_name,
            "question": row.get("question"),
            "conversation_a": row.get("conversation_a"),
            "conversation_b": row.get("conversation_b"),
            "reference_answer": row.get("reference_answer"),
        }

        if row_idx is not None:
            if row_idx in map_by_row_idx and map_by_row_idx[row_idx].get("dataset") != dataset_name:
                row_idx_conflicts.setdefault(row_idx, set()).update({map_by_row_idx[row_idx].get("dataset"), dataset_name})
            else:
                map_by_row_idx[row_idx] = meta

        if question_id is not None:
            if (
                question_id in map_by_question_id
                and map_by_question_id[question_id].get("dataset") != dataset_name
            ):
                question_id_conflicts.setdefault(question_id, set()).update(
                    {map_by_question_id[question_id].get("dataset"), dataset_name}
                )
            else:
                map_by_question_id[question_id] = meta

    return map_by_row_idx, map_by_question_id, row_idx_conflicts, question_id_conflicts


def print_conversation(conversation, title, width=105):
    print(title)
    if not isinstance(conversation, list) or not conversation:
        print("  <missing>")
        print("-" * 80)
        return

    valid_turn_count = 0
    for idx, turn in enumerate(conversation, start=1):
        if not isinstance(turn, dict):
            continue
        role = normalize_text(turn.get("role"), default="unknown")
        content = wrap_long_text(turn.get("content"), width=width)
        print(f"  {idx}. [{role}]")
        for line in str(content).splitlines():
            print(f"     {line}")
        valid_turn_count += 1

    if valid_turn_count == 0:
        print("  <missing>")
    print("-" * 80)


required_columns = {"question_id", "temperature"}
missing_required = required_columns - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns for case study: {sorted(missing_required)}")

mapping_path = Path("input/combined_dataset_with_reference_good_row_idx.json")
mapping_records = load_mapping_records(mapping_path)
(
    map_by_row_idx,
    map_by_question_id,
    row_idx_conflicts,
    question_id_conflicts,
) = build_mapping_indices(mapping_records)

df = df.copy()
df["question_id"] = pd.to_numeric(df["question_id"], errors="coerce").astype("Int64")

result_question_ids = sorted(df["question_id"].dropna().astype(int).unique().tolist())
result_qid_set = set(result_question_ids)

row_idx_overlap = len(result_qid_set & set(map_by_row_idx.keys()))
question_id_overlap = len(result_qid_set & set(map_by_question_id.keys()))

if row_idx_overlap >= question_id_overlap:
    active_map = map_by_row_idx
    active_key_name = "row_idx"
else:
    active_map = map_by_question_id
    active_key_name = "question_id"


def get_meta_by_active_key(value):
    if pd.isna(value):
        return None
    return active_map.get(int(value))


mapped_meta = df["question_id"].map(get_meta_by_active_key)
mapped_dataset = mapped_meta.map(
    lambda meta: meta.get("dataset") if isinstance(meta, dict) else pd.NA
)

if "dataset" in df.columns:
    df["dataset"] = mapped_dataset.fillna(df["dataset"])
else:
    df["dataset"] = mapped_dataset

mapped_source_question_id = mapped_meta.map(
    lambda meta: meta.get("source_question_id") if isinstance(meta, dict) else pd.NA
)
df["source_question_id"] = pd.to_numeric(mapped_source_question_id, errors="coerce").astype("Int64")

if df["dataset"].dropna().empty:
    dataset_options = ["all"]
else:
    dataset_options = ["all"] + sorted(df["dataset"].dropna().astype(str).unique().tolist())

question_options_all = sorted(df["question_id"].dropna().astype(int).unique().tolist())
repeat_options = (
    sorted(df["repeat_id"].dropna().astype(int).unique().tolist())
    if "repeat_id" in df.columns
    else [0]
)
judge_options = (
    sorted(df["judge_type"].dropna().astype(str).unique().tolist())
    if "judge_type" in df.columns
    else ["all"]
)
prompt_options = (
    sorted(df["prompt_variant"].dropna().astype(str).unique().tolist())
    if "prompt_variant" in df.columns
    else ["all"]
)

active_key_set = set(active_map.keys())
unmatched_question_ids = sorted(result_qid_set - active_key_set)
unused_mapping_ids = sorted(active_key_set - result_qid_set)

if not question_options_all:
    raise ValueError("No question_id values available after loading data.")


case_df = pd.DataFrame()


def get_question_options_for_dataset(dataset_name):
    subset = df if dataset_name == "all" else df[df["dataset"] == dataset_name]
    return sorted(subset["question_id"].dropna().astype(int).unique().tolist())


def show_case_study(dataset_name, question_id, repeat_id, judge_type, prompt_variant):
    global case_df

    if question_id is None:
        case_df = pd.DataFrame()
        display(Markdown("当前 dataset 下没有可用 question。"))
        return

    meta = active_map.get(int(question_id))

    subset = df.copy()
    if "dataset" in subset.columns and dataset_name != "all":
        subset = subset[subset["dataset"] == dataset_name]

    subset = subset[subset["question_id"] == question_id]

    if "repeat_id" in subset.columns:
        subset = subset[subset["repeat_id"] == repeat_id]
    if "judge_type" in subset.columns and judge_type != "all":
        subset = subset[subset["judge_type"] == judge_type]
    if "prompt_variant" in subset.columns and prompt_variant != "all":
        subset = subset[subset["prompt_variant"] == prompt_variant]

    subset["temperature"] = pd.to_numeric(subset["temperature"], errors="coerce")
    subset = subset.dropna(subset=["temperature"]).sort_values("temperature")

    case_df = subset.drop_duplicates(subset=["temperature"], keep="first").copy()
    case_df = case_df.sort_values("temperature")

    selected_dataset = dataset_name
    if selected_dataset == "all" and "dataset" in case_df.columns and not case_df.empty:
        unique_dataset_names = sorted(case_df["dataset"].dropna().astype(str).unique().tolist())
        selected_dataset = ", ".join(unique_dataset_names) if unique_dataset_names else "all"

    display(Markdown(f"### Case Study | Dataset: {selected_dataset} | Question: {question_id}"))

    if isinstance(meta, dict):
        source_qid = meta.get("source_question_id")
        source_row_idx = meta.get("row_idx")
        display(Markdown(f"#### Prompt / Conversation（从映射文件按 **{active_key_name}** 匹配）"))

        info_bits = []
        if source_row_idx is not None:
            info_bits.append(f"row_idx = {source_row_idx}")
        if source_qid is not None:
            info_bits.append(f"source question_id = {source_qid}")
        if info_bits:
            display(Markdown("- " + " | ".join(info_bits)))

        question_field = normalize_text(meta.get("question"))
        if question_field != "<missing>":
            print("Question Field:")
            print(wrap_long_text(question_field, width=110))
            print("-" * 80)

        print_conversation(meta.get("conversation_a"), "Conversation A:", width=110)
        print_conversation(meta.get("conversation_b"), "Conversation B:", width=110)
    else:
        display(Markdown("> 映射文件中未找到该 question 的会话信息。"))

    if case_df.empty:
        display(Markdown("没有找到匹配记录，请调整筛选条件。"))
        return

    temperatures = case_df["temperature"].tolist()
    temp_list = ", ".join(str(t) for t in temperatures)
    display(Markdown(f"- 温度点数量: **{len(temperatures)}** ({temp_list})"))
    if len(temperatures) != 6:
        display(Markdown(f"> 当前匹配到 **{len(temperatures)}** 个温度点，不是 6 个。"))

    for _, row in case_df.iterrows():
        parsed_output = parse_raw_output(row.get("raw_output"))

        judge_reason = row.get("judge_reason")
        if normalize_text(judge_reason) == "<missing>":
            judge_reason = parsed_output.get("judge_reason")

        judge_result = parsed_output.get("judge_result", row.get("pairwise_winner"))

        print(f"Dataset: {row.get('dataset', selected_dataset)}")
        print(f"Question ID: {row.get('question_id')}, Repeat ID: {row.get('repeat_id')}")
        if pd.notna(row.get("source_question_id")):
            print(f"Mapped Source Question ID: {int(row.get('source_question_id'))}")
        print(f"Temperature: {row.get('temperature')}")
        print(f"Judge Result: {normalize_text(judge_result)}")
        print("Judge Reason:")
        print(wrap_long_text(judge_reason, width=100))
        print("-" * 80)


dataset_widget = widgets.Dropdown(
    options=dataset_options,
    value=dataset_options[0],
    description="Dataset",
    layout=widgets.Layout(width="320px"),
)

initial_question_options = get_question_options_for_dataset(dataset_widget.value)
if not initial_question_options:
    initial_question_options = question_options_all

question_widget = widgets.Dropdown(
    options=initial_question_options,
    value=initial_question_options[0],
    description="Question",
    layout=widgets.Layout(width="280px"),
)

repeat_widget = widgets.Dropdown(
    options=repeat_options,
    value=repeat_options[0],
    description="Repeat",
    layout=widgets.Layout(width="220px"),
)
judge_widget = widgets.Dropdown(
    options=judge_options,
    value=judge_options[0],
    description="Judge",
    layout=widgets.Layout(width="280px"),
)
prompt_widget = widgets.Dropdown(
    options=prompt_options,
    value=prompt_options[0],
    description="Prompt",
    layout=widgets.Layout(width="320px"),
)


def on_dataset_change(change):
    if change.get("name") != "value":
        return

    new_options = get_question_options_for_dataset(change["new"])
    if not new_options:
        question_widget.options = []
        question_widget.value = None
        return

    current_question = question_widget.value
    question_widget.options = new_options
    question_widget.value = current_question if current_question in new_options else new_options[0]


dataset_widget.observe(on_dataset_change, names="value")

controls_top = widgets.HBox([dataset_widget, question_widget, repeat_widget])
controls_bottom = widgets.HBox([judge_widget, prompt_widget])

output = widgets.interactive_output(
    show_case_study,
    {
        "dataset_name": dataset_widget,
        "question_id": question_widget,
        "repeat_id": repeat_widget,
        "judge_type": judge_widget,
        "prompt_variant": prompt_widget,
    },
)

mapped_rows = int(df["dataset"].notna().sum())
matched_key_count = len(result_qid_set) - len(unmatched_question_ids)

display(Markdown("使用下拉框选择 dataset 和 question，查看该样本在不同温度下的回答。"))
display(Markdown(f"- 自动匹配键：**{active_key_name}**"))
display(Markdown(f"- 覆盖对比：`row_idx` = **{row_idx_overlap}/{len(result_qid_set)}**，`question_id` = **{question_id_overlap}/{len(result_qid_set)}**"))
display(Markdown(f"- 已按映射文件匹配 dataset：**{mapped_rows}/{len(df)}** 行；匹配到 **{matched_key_count}/{len(result_qid_set)}** 个 question"))

if active_key_name == "row_idx" and question_id_overlap < len(result_qid_set):
    display(Markdown(
        "> 未匹配完整的根因：当前结果文件里的 `question_id` 实际对应映射文件的 `row_idx`，而不是映射文件里的 `question_id`。"
    ))

if unmatched_question_ids:
    display(Markdown(f"> 仍有 {len(unmatched_question_ids)} 个 question 未匹配。示例：{unmatched_question_ids[:20]}"))
if unused_mapping_ids:
    display(Markdown(f"> 映射文件里有 {len(unused_mapping_ids)} 个键当前结果文件未使用。示例：{unused_mapping_ids[:20]}"))
if row_idx_conflicts:
    display(Markdown(f"> 映射文件按 row_idx 有 {len(row_idx_conflicts)} 个 dataset 冲突，当前使用首次出现值。"))
if question_id_conflicts:
    display(Markdown(f"> 映射文件按 question_id 有 {len(question_id_conflicts)} 个 dataset 冲突，当前使用首次出现值。"))

display(widgets.VBox([controls_top, controls_bottom]), output)

使用下拉框选择 dataset 和 question，查看该样本在不同温度下的回答。

- 自动匹配键：**row_idx**

- 覆盖对比：`row_idx` = **500/500**，`question_id` = **86/500**

- 已按映射文件匹配 dataset：**179998/179998** 行；匹配到 **500/500** 个 question

> 未匹配完整的根因：当前结果文件里的 `question_id` 实际对应映射文件的 `row_idx`，而不是映射文件里的 `question_id`。

> 映射文件按 question_id 有 1 个 dataset 冲突，当前使用首次出现值。

Output()